# JVP, VJP

This notebook contains implementations of the Jacobian–Vector Product (JVP, or linear tangent) and the Vector–Jacobian Product (VJP) applied to a loss function. It begins by demonstrating these concepts using a manual gradient-descent approach to illustrate the principles of automatic differentiation. It then shows how to perform the same operations using `JAX`, including ways to optimize the computation with JAX’s built-in differentiation tools.

The loss function used is taken from the following notebook:
https://github.com/Cambridge-ICCS/differentiable-programming-summer-school-2025/blob/main/session1/notebook.ipynb

In [1]:
import ast
import jax
import jax.numpy as jnp
from jax import jit, lax
import numpy as np
import time
import functools

In [62]:
jax.config.update('jax_enable_x64', True)

In [2]:
import sys, os
sys.path.append(os.path.abspath(".."))

In [ ]:
%reload_ext autoreload
%autoreload 2
from fgpt.core.transpiler import F2NP
from fgpt.core.frontend import Processor
from fgpt.core.common import Logger

In [4]:
logger = Logger()
processor = Processor(logger=logger)

In [5]:
f2np_ = F2NP()

╭────────────────────────────────────── Fortran General Purpose Transformer ──────────────────────────────────────╮
│ 🚀 Starting Module: F2NP                                                                                        │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

## Gradient based optimization

In [6]:
# This is the cost function
cost_function  = """
 SUBROUTINE COST_FUNCTION(u, j)
    IMPLICIT NONE
! Numerical solution at the end time
    REAL, INTENT(IN) :: u
! Cost function value
    REAL, INTENT(OUT) :: j
    INTRINSIC EXP
! The exponential constant, e=2.71828...
    REAL, PARAMETER :: e=EXP(1.0)
    j = (u-e)**2
  END SUBROUTINE COST_FUNCTION
"""

cost_func = processor.parse_fortran_string(cost_function)
print(cost_func)

[INFO] Successfully parsed string!


SUBROUTINE COST_FUNCTION(u, j)
  IMPLICIT NONE
  ! Numerical solution at the end time
  REAL, INTENT(IN) :: u
  ! Cost function value
  REAL, INTENT(OUT) :: j
  INTRINSIC :: EXP
  ! The exponential constant, e=2.71828...
  REAL, PARAMETER :: e = EXP(1.0)
  j = (u - e) ** 2
END SUBROUTINE COST_FUNCTION


In [7]:
_,_,cost_func_python = f2np_.recursive_ast(cost_func)

In [8]:
print(ast.unparse(ast.fix_missing_locations(cost_func_python[0])))

def COST_FUNCTION(u, j):
    e = np.float64(np.exp(1.0))
    j = (u - e) ** 2


Tapenade provides the tangent version of the cost function, which contains the
propagated derivative variables required for forward-mode automatic
differentiation. We then translate this generated code into Python so that it
can be integrated into our workflow.

In [9]:
cost_function_tangent = """
  SUBROUTINE COST_FUNCTION_D(u, ud, j, jd)
    IMPLICIT NONE
! Numerical solution at the end time
    REAL, INTENT(IN) :: u
    REAL, INTENT(IN) :: ud
! Cost function value
    REAL, INTENT(OUT) :: j
    REAL, INTENT(OUT) :: jd
    INTRINSIC EXP
! The exponential constant, e=2.71828...
    REAL, PARAMETER :: e=EXP(1.0)
    jd = 2*(u-e)*ud
    j = (u-e)**2
  END SUBROUTINE COST_FUNCTION_D
"""


cost_func_tangent = processor.parse_fortran_string(cost_function_tangent)
print(cost_func_tangent)

[INFO] Successfully parsed string!


SUBROUTINE COST_FUNCTION_D(u, ud, j, jd)
  IMPLICIT NONE
  ! Numerical solution at the end time
  REAL, INTENT(IN) :: u
  REAL, INTENT(IN) :: ud
  ! Cost function value
  REAL, INTENT(OUT) :: j
  REAL, INTENT(OUT) :: jd
  INTRINSIC :: EXP
  ! The exponential constant, e=2.71828...
  REAL, PARAMETER :: e = EXP(1.0)
  jd = 2 * (u - e) * ud
  j = (u - e) ** 2
END SUBROUTINE COST_FUNCTION_D


In [10]:
_,_,cost_func_python_tangent = f2np_.recursive_ast(cost_func_tangent)

In [ ]:
print(ast.unparse(ast.fix_missing_locations(cost_func_python_tangent[0])))
# The generated tangent code closely resembles the original function. Rather
# than rewriting the entire algorithm, Tapenade augments it by introducing
# tangent variables (e.g., `ud` and `jd`) that propagate derivatives alongside
# the primal variables. This is the essence of forward-mode automatic
# differentiation.

def COST_FUNCTION_D(u, ud, j, jd):
    e = np.float64(np.exp(1.0))
    jd = 2 * (u - e) * ud
    j = (u - e) ** 2


In [12]:
# Now we need to define the time step
def theta_method(theta,u):
    end_time = 1.0
    dt = 0.1 # timestep
    t = 0.0
    u_ = 1.0

    while(t < end_time - 1e-05):
        u = u_ * (1 + dt * (1 - theta)) / (1 - dt * theta)
        u_ = u
        t = t + dt
    
    return u 

def theta_method_d(theta,thetad):
    end_time = 1.0
    dt = 0.1 # timestep
    t = 0.0
    u_ = 1.0
    u_d = 0.0
    ud = 0.0 
    while(t < end_time - 1e-05):
        temp = u_/(-(dt*theta)+1)
        ud = (dt*(1-theta)+1)*(u_d+temp*dt*thetad)/(1-dt*theta) - temp*dt*thetad
        u = (dt*(1-theta)+1)*temp
        u_d = ud
        u_ = u
        t = t + dt
    return u,ud


In [ ]:
# since these functions are subroutine that are translated to function definition we need to add the return statement
# since the transformer class is the one that occupies of this. 
def COST_FUNCTION(u):
    e = np.exp(1.0)
    j = (u - e) ** 2
    return j

def COST_FUNCTION_D(u, ud):
    e = np.exp(1.0)
    jd = 2 * (u - e) * ud
    j = (u - e) ** 2
    return j,jd

In [14]:
def timer(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        start_time = time.perf_counter() 
        result = func(*args, **kwargs)
        end_time = time.perf_counter()
        duration = end_time - start_time
        print(f"[TIMER] '{func.__name__}' executed in {duration:.6f} seconds")
        return result
    return wrapper

# Gradient Descent Algorithm

This notebook implements a simple **gradient descent** optimization algorithm to minimize the cost function \(J(\theta)\).

## Parameters

- **`maxiter = 1000`**: Maximum number of iterations.
- **`gtol = 1e-5`**: Gradient convergence tolerance. Optimization stops when the gradient changes by less than this value.
- **`dtol = 1.1`**: Divergence tolerance based on the normalized cost function.
- **`alpha = 0.2`**: Gradient descent learning rate (step size).

## Algorithm

1. Initialize the optimization variable:

   $\theta = 0, \qquad \dot{\theta} = 1$


2. At each iteration:

   - Evaluate the model and its derivative:

     $(u, \dot{u}) = \texttt{theta\_method\_d}(\theta, \dot{\theta})$

   - Evaluate the cost function and its derivative:

     $(J, J') = \texttt{COST\_FUNCTION\_D}(u, \dot{u})$

3. Check stopping criteria:

   - Maximum number of iterations reached.
   - Gradient convergence:

     $|J'_k - J'_{k-1}| < \texttt{gtol}$

   - Cost function divergence:

     $\left|\frac{J_k}{J_0}\right| > \texttt{dtol}$

4. Update the search direction using steepest descent:

   $p = -J'$

5. Update the optimization variable:

   $\theta_{k+1} = \theta_k + \alpha p$


## Notes

- The implementation uses a **fixed learning rate** (`alpha`).
- The search direction is simply the negative gradient, corresponding to the classical steepest descent method.
- Diagnostic information is printed at each iteration, including:
  - iteration number,
  - current value of \(\theta\),
  - state variables \((u, \dot{u})\),
  - cost \(J\),
  - gradient \(J'\),
  - change in the gradient.

In [15]:
@timer
def gradient_descent():
    maxiter = 1000
    gtol = 1e-05
    dtol = 1.1
    alpha = 0.2 # temporary variables 
    theta, thetad = 0.0, 1.0
    jd_ = 1.
    for i in range(maxiter + 1):

        if i == maxiter:
            print(f'Reached maximum iterations without convergence')
            break

        u, ud = theta_method_d(theta,thetad)
        j, jd = COST_FUNCTION_D(u,ud)

        if i == 0:
            j_init = j
        elif np.abs(jd - jd_) < gtol:
            print(f'Converged in {i} iterations due to gradient convergence')
            break
        elif np.abs(j/j_init) > dtol:
            print(f'Detected cost function divergence after {i} iterations')
            return
        
        print(f'Iteration n°{i}, theta = {theta}, u = {u}, ud = {ud}, j = {j}, jd = {jd}, abs diff = {np.abs(jd/jd_)}')
        
        jd_ = jd
        p = -jd
        theta = theta + alpha * p


gradient_descent()


Iteration n°0, theta = 0.0, u = 2.5937424601000023, ud = 0.2357947691000004, j = 0.015510054271269346, jd = -0.05873146321216078, abs diff = 0.05873146321216078
Iteration n°1, theta = 0.011746292642432156, u = 2.596516766118851, ud = 0.236577200226463, j = 0.01482673040671139, jd = -0.05761367506768777, abs diff = 0.9809678137860259
Iteration n°2, theta = 0.02326902765596971, u = 2.5992472221770275, ud = 0.23734839914342687, j = 0.014169237492714934, jd = -0.056505346487409944, abs diff = 0.9807627515693853
Iteration n°3, theta = 0.0345700969534517, u = 2.6019338034391923, ud = 0.23810830403116778, j = 0.013536862926020281, jd = -0.05540686182970603, abs diff = 0.9805596332738412
Iteration n°4, theta = 0.04565146931939291, u = 2.604576514603927, ud = 0.23885686232703573, j = 0.012928898398890954, jd = -0.054318588994688756, abs diff = 0.9803585187993124
Iteration n°5, theta = 0.05651518711833066, u = 2.6071753893667147, ud = 0.23959403063030288, j = 0.01234464080777773, jd = -0.0532408

In [16]:
%timeit -n 1 -r 1 gradient_descent()

Iteration n°0, theta = 0.0, u = 2.5937424601000023, ud = 0.2357947691000004, j = 0.015510054271269346, jd = -0.05873146321216078, abs diff = 0.05873146321216078
Iteration n°1, theta = 0.011746292642432156, u = 2.596516766118851, ud = 0.236577200226463, j = 0.01482673040671139, jd = -0.05761367506768777, abs diff = 0.9809678137860259
Iteration n°2, theta = 0.02326902765596971, u = 2.5992472221770275, ud = 0.23734839914342687, j = 0.014169237492714934, jd = -0.056505346487409944, abs diff = 0.9807627515693853
Iteration n°3, theta = 0.0345700969534517, u = 2.6019338034391923, ud = 0.23810830403116778, j = 0.013536862926020281, jd = -0.05540686182970603, abs diff = 0.9805596332738412
Iteration n°4, theta = 0.04565146931939291, u = 2.604576514603927, ud = 0.23885686232703573, j = 0.012928898398890954, jd = -0.054318588994688756, abs diff = 0.9803585187993124
Iteration n°5, theta = 0.05651518711833066, u = 2.6071753893667147, ud = 0.23959403063030288, j = 0.01234464080777773, jd = -0.0532408

In [ ]:
# We can implement the same algorithm in JAX without using `@jit`. Although this
# version still benefits from JAX's automatic differentiation (`jax.grad`), the
# functions are interpreted on each call rather than being compiled ahead of time.

def COST_FUNCTION(u):
    e = jnp.exp(1.0)
    j = (u - e) ** 2
    return j


def theta_step(u, theta, dt):
    return u * (1 + dt * (1 - theta)) / (1 - dt * theta)


# As in the JIT version, the time-stepping loop is expressed using `jax.lax.scan`
# instead of a Python loop. `scan` is well suited for fixed-length iterative
# computations and is compatible with JAX transformations such as automatic
# differentiation. Here, the number of iterations is known in advance.

def theta_method(theta):
    end_time = 1.0
    dt = 0.1  # Time step
    u_ = 1.0

    N = int(end_time / dt)  # Number of time steps

    def body(u, _):
        return theta_step(u, theta, dt), None

    u_final, _ = jax.lax.scan(body, u_, jnp.arange(N))
    return u_final


# Define the scalar-valued objective function with respect to `theta`.
def cost_grad(theta):
    u = theta_method(theta)
    return COST_FUNCTION(u)


# Compute the gradient using JAX automatic differentiation.
grad_cost = jax.grad(cost_grad)

In [18]:
@timer
def gradient_descent_jax(maxiter = 1000, gtol= 1e-05, alpha = 0.2, theta = 0.0):
    jd_ = 1.0
    for i in range(maxiter):
        if i == maxiter:
            print(f'Reached maximum iterations without convergence')
            break
        
        jd = grad_cost(theta)
        theta = theta - alpha * jd
        
        if np.abs(jd - jd_) < gtol:
            print(f'Converged in {i} iterations due to gradient convergence')
            break

        jd_ = jd

# %timeit -n 1 -r 1 gradient_descent_jax() 
# %timeit -n 1 -r 1 gradient_descent_jax()

# As seen in the results below, the execution time is much extremely much larger than than that of the python version

In [ ]:
# We can implement the same cost function using JAX. By decorating functions with
# `@jit`, JAX compiles them once (on the first call) and reuses the compiled version
# for subsequent calls, significantly reducing execution time. This requires replacing
# NumPy (`np`) with JAX NumPy (`jnp`).

@jit
def COST_FUNCTION(u):
    e = jnp.exp(1.0)
    j = (u - e) ** 2
    return j

# Once the function is written using JAX operations, its derivative can be obtained
# automatically with `jax.grad`. One important difference from NumPy is that JAX
# traces functions before compiling them. Therefore, Python control flow (e.g.,
# `for` and `while` loops) is often replaced with JAX primitives such as
# `jax.lax.scan`, `jax.lax.fori_loop`, or `jax.lax.while_loop`, which are compatible
# with JIT compilation.

def theta_step(u, theta, dt):
    return u * (1 + dt * (1 - theta)) / (1 - dt * theta)

# Here, the time-stepping loop is implemented using `jax.lax.scan`. Since only the
# solution `u` is updated at each iteration, `scan` provides an efficient way to
# express the recurrence. It is particularly well suited for fixed-length iterations
# and allows JAX to optimize the entire loop during compilation.

@jit
def theta_method(theta):
    end_time = 1.0
    dt = 0.1
    u_ = 1.0

    N = int(end_time / dt)

    def body(u, _):
        return theta_step(u, theta, dt), None

    u_final, _ = jax.lax.scan(body, u_, jnp.arange(N))
    return u_final

# Define a scalar-valued function whose gradient will be computed with respect to
# `theta`.

def cost_grad(theta):
    u = theta_method(theta)
    return COST_FUNCTION(u)

# Compute and JIT-compile the gradient function.
grad_cost = jit(jax.grad(cost_grad))

In [ ]:
@timer
def gradient_descent_jax(maxiter = 1000, gtol= 1e-05, alpha = 0.2, theta = 0.0):
    jd_ = 1.0
    for i in range(maxiter):
        if i == maxiter:
            print(f'Reached maximum iterations without convergence')
            break
        
        jd = grad_cost(theta)
        theta = theta - alpha * jd
        
        if np.abs(jd - jd_) < gtol:
            print(f'Converged in {i} iterations due to gradient convergence')
            break

        jd_ = jd

# The first call includes the JIT compilation overhead, so it is expected to take
# longer. During this execution, JAX traces the function, compiles it, and then
# executes the compiled code.

%timeit -n 1 -r 1 gradient_descent_jax()

# On subsequent calls, the previously compiled version is reused. Since no
# recompilation is required (provided the input shapes and data types remain the
# same), only the execution time is measured, making the function significantly
# faster.

%timeit -n 1 -r 1 gradient_descent_jax()

Converged in 186 iterations due to gradient convergence
[TIMER] 'gradient_descent_jax' executed in 1.063876 seconds
1.06 s ± 0 ns per loop (mean ± std. dev. of 1 run, 1 loop each)
Converged in 186 iterations due to gradient convergence
[TIMER] 'gradient_descent_jax' executed in 0.008032 seconds
8.04 ms ± 0 ns per loop (mean ± std. dev. of 1 run, 1 loop each)


Instead of using `jax.lax.scan`, we could have implemented the time-stepping loop with `jax.lax.while_loop`. However, `lax.while_loop` is **not reverse-mode differentiable**, so it cannot be used directly with transformations such as `jax.grad`. For this reason, `lax.scan` is generally preferred for fixed-length iterative computations, as it supports both forward- and reverse-mode automatic differentiation.

For more details, see the JAX documentation on structured control-flow primitives:
https://docs.jax.dev/en/latest/control-flow.html#structured-control-flow-primitives

# JVP, VJP

Since we have already implemented the forward-mode approach using JVP, we will
now focus on the reverse-mode implementation using VJP.

In [ ]:
fortran_F = """
SUBROUTINE F(x, y)
    IMPLICIT NONE
    REAL, DIMENSION(2), INTENT(IN) :: x
    REAL, INTENT(OUT) :: y
    y = x(1)*x(2)
  END SUBROUTINE F
"""
fortran_F_code = processor.parse_fortran_string(fortran_F)

[INFO] Successfully parsed string!

In [ ]:
_,_,python_F_code = f2np_.recursive_ast(fortran_F_code)
print(ast.unparse(ast.fix_missing_locations(python_F_code[0]))) 

def F(x, y):
    y = x[1] * x[2]


In [27]:
fortran_F_vjp = """
SUBROUTINE F_B(x, xb, y, yb)
    IMPLICIT NONE
    REAL, DIMENSION(2), INTENT(IN) :: x
    REAL, DIMENSION(2) :: xb
    REAL :: y
    REAL :: yb
    xb = 0.0
    xb(1) = xb(1) + x(2)*yb
    xb(2) = xb(2) + x(1)*yb
    yb = 0.0
  END SUBROUTINE F_B
"""

fortran_F_vjp_code = processor.parse_fortran_string(fortran_F_vjp)

[INFO] Successfully parsed string!

In [28]:
_,_,python_F_code = f2np_.recursive_ast(fortran_F_vjp_code)
print(ast.unparse(ast.fix_missing_locations(python_F_code[0]))) 

def F_B(x, xb, y, yb):
    xb = 0.0
    xb[1] = xb[1] + x[2] * yb
    xb[2] = xb[2] + x[1] * yb
    yb = 0.0


In [ ]:
def F(x): # Original function 
    y = x[0] * x[1]
    return y

def F_D(x, xd): # Forward mode (tangent mode)
    yd = x[0] * xd[1] + x[1] * xd[0]
    return yd

def F_B(x,xb,yb): # VECTOR-Jacobian PRODUCT(adjoint mode)
    xb[0] = xb[0] + x[1] * yb
    xb[1] = xb[1] + x[0] * yb
    return xb 


def dot_product_test():
    atol = 1e-05
    x = jnp.array([1.2, -2.3], dtype=jnp.float64) # Primal input 
    xd = jnp.array([4.2, -0.7], dtype=jnp.float64) # Tangent seed
    xb = np.zeros((2,),dtype=np.float64)

    yd = F_D(x,xd)
    yb = 3.0 # reverse mode seed 
    result1 = np.dot([yd],[yb])
    xb[:] = [0.0, 0.0]
    xb = F_B(x,xb,yb)
    result2 = np.dot(xd,xb)

    if (np.abs(result1 - result2) < atol):
        print(f'PASS')
    else:
        print(f"FAIL with atol = {atol}")

dot_product_test()


PASS


In [ ]:
# We can also verify the dot-product (adjoint) identity using JAX's forward-mode
# (`jax.jvp`) and reverse-mode (`jax.vjp`) automatic differentiation.

def dot_product_jax_test():
    atol = 1e-5

    # Primal input and tangent (directional derivative).
    x = jnp.array([1.2, -2.3], dtype=jnp.float64)
    xd = jnp.array([4.2, -0.7], dtype=jnp.float64)

    # Compute the Jacobian-vector product (JVP).
    _, tangent = jax.jvp(F, (x,), (xd,))

    # Reverse-mode seed (adjoint).
    yb = 3.0

    # <J xd, yb>
    result1 = jnp.dot(tangent, yb)

    # Compute the vector-Jacobian product (VJP). Unlike `jax.jvp`, `jax.vjp`
    # returns the function output and a pullback (adjoint) function. Applying
    # the pullback to the output seed `yb` computes J^T yb.
    _, vjp_fn = jax.vjp(F, x)

    # <xd, J^T yb>
    result2 = jnp.dot(xd, vjp_fn((yb,))[0])

    # Verify the dot-product identity:
    # <J xd, yb> = <xd, J^T yb>
    # which should hold up to numerical precision.
    if jnp.abs(result1 - result2) < atol:
        print("PASS")
    else:
        print(f"FAIL with atol = {atol}")

dot_product_jax_test()

PASS
